# 04 - CaVE for Binary Linear Programs

This notebook introduces **CaVE** (Cone-Aligned Vector Estimation), a fast
end-to-end loss for predict-then-optimize problems where the inner
optimization is a **binary linear program (BLP)** - shortest path, knapsack,
TSP, vehicle routing, etc.

> Reference: Tang & Khalil, *CaVE: A Cone-Aligned Approach for Fast
> Predict-then-optimize with Binary Linear Programs*, CPAIOR 2024.
> Paper: <https://arxiv.org/abs/2312.07718> Repo: <https://github.com/khalil-research/CaVE>


## 1. The idea in one paragraph

Most decision-focused losses (SPO+, DPO, PFYL, ...) call the solver **once
per training step** to turn the predicted cost into a decision. That
per-step solve is the bottleneck at scale.

**CaVE skips it.** For each training instance, the true optimal vertex is
the intersection of a small set of *binding* (tight) constraints. The
cost vectors that pick that vertex form a polyhedral cone - the normal
cone at the optimum. CaVE pre-computes the binding-constraint normals
once during data prep, then at every training step it just projects the
predicted cost onto that cone and minimizes
$1 - \cos(-\hat{\mathbf{c}}, \mathrm{proj})$.

No solver call per step. The projection is a small, batched APGD that runs
natively on GPU.

**CaVE vs CaVE+.** The paper proposes two presets:
- **CaVE** runs APGD to a tolerance (slower, lands on the cone boundary).
- **CaVE+** truncates APGD at 20 iterations (faster, lands strictly inside
  the cone, no quality loss in practice). This is what `coneAlignedCosine`
  uses by default.

**When to use it.**
- The inner problem must be a **binary LP** (sols in {0,1}^n). Continuous
  or general integer problems are out of scope.
- The headline speedup is realized on **GPU**. The projection is a
  `torch.compile`'d APGD designed for CUDA - on CPU the compile overhead
  dominates and CaVE can end up no faster than (or slower than) SPO+.


## 2. Setup

Standard imports plus a device check. We seed everything so the runs are
reproducible.


In [ ]:
import time
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

import pyepo
from pyepo.data.dataset import optDataset, optDatasetConstrs, collate_tight_constraints
from pyepo.model.grb import tspDFJModel

torch.manual_seed(0)
np.random.seed(0)
if torch.cuda.is_available():
    torch.cuda.manual_seed(0)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device = {device}")
if device.type != "cuda":
    print("[warning] CaVE+ is designed for GPU; on CPU it will not be faster than SPO+.")


## 3. The problem: TSP-DFJ

Travelling Salesman Problem with **DFJ subtour elimination** (lazy
constraints). Binary edge variables, so it is a textbook BLP. We use
the **paper's hyperparameters** consistently:

| setting | value |
|---|---|
| `num_data` (train + test) | 1000 + 1000 |
| `num_feat` | 10 |
| `deg` | 4 |
| `noise_width` | 0.5 |

These match `code_sample.py` in the CaVE repo and the experimental setup
in the paper.


In [ ]:
# global hyperparameters shared across all TSP sizes
NUM_DATA = 1000
NUM_TEST = 1000
NUM_FEAT = 10
DEG = 4
NOISE = 0.5

BATCH = 32
EPOCHS = 10
LR = 1e-2


## 4. Two datasets, one optimization model

SPO+ and CaVE consume slightly different datasets:

- **`optDataset`** - features, true costs, true solutions, true objectives.
  This is what SPO+, DPO, blackbox, etc. all use.
- **`optDatasetConstrs`** - the same four items **plus** the matrix of
  tight-constraint normals at each optimum. CaVE needs this to project.

Building `optDatasetConstrs` is a bit slower than `optDataset` because it
also extracts the active set after each solve. You pay this cost once.


## 5. Predictor

A tiny linear model that maps a feature vector to a per-edge cost. The
same network is used for SPO+ and for CaVE+ so the only thing that
differs across runs is the loss.


In [ ]:
class LinearPred(nn.Module):
    def __init__(self, num_feat, num_cost):
        super().__init__()
        self.linear = nn.Linear(num_feat, num_cost)

    def forward(self, x):
        return self.linear(x)


## 6. Training loops

Two near-identical loops. The CaVE loop just receives one extra item per
batch (`tight_ctrs`) and feeds it to `coneAlignedCosine` instead of
`SPOPlus`. Everything else - optimizer, predictor, learning rate - is
shared.

We also measure wall-clock training time (excluding dataset construction).


In [ ]:
def regret(predmodel, optmodel, loader):
    return pyepo.metric.regret(predmodel, optmodel, loader)


def train_spo(optmodel, train_loader, test_loader, num_feat, num_epochs=EPOCHS, lr=LR):
    pred = LinearPred(num_feat, optmodel.num_cost).to(device)
    opt = torch.optim.Adam(pred.parameters(), lr=lr)
    loss_fn = pyepo.func.SPOPlus(optmodel, processes=1)
    regrets = [regret(pred, optmodel, test_loader)]
    if device.type == "cuda":
        torch.cuda.synchronize()
    t0 = time.time()
    for ep in range(num_epochs):
        for x, c, w, z in train_loader:
            x, c, w, z = x.to(device), c.to(device), w.to(device), z.to(device)
            cp = pred(x)
            loss = loss_fn(cp, c, w, z)
            opt.zero_grad(); loss.backward(); opt.step()
        regrets.append(regret(pred, optmodel, test_loader))
    if device.type == "cuda":
        torch.cuda.synchronize()
    return regrets, time.time() - t0


def train_cave(optmodel, train_loader, test_loader, num_feat, num_epochs=EPOCHS, lr=LR):
    pred = LinearPred(num_feat, optmodel.num_cost).to(device)
    opt = torch.optim.Adam(pred.parameters(), lr=lr)
    # defaults reproduce the CaVE+ truncated-APGD preset
    loss_fn = pyepo.func.coneAlignedCosine(optmodel, processes=1)
    regrets = [regret(pred, optmodel, test_loader)]
    if device.type == "cuda":
        torch.cuda.synchronize()
    t0 = time.time()
    for ep in range(num_epochs):
        for x, c, w, z, tight_ctrs in train_loader:
            x, tight_ctrs = x.to(device), tight_ctrs.to(device)
            cp = pred(x)
            loss = loss_fn(cp, tight_ctrs)
            opt.zero_grad(); loss.backward(); opt.step()
        regrets.append(regret(pred, optmodel, test_loader))
    if device.type == "cuda":
        torch.cuda.synchronize()
    return regrets, time.time() - t0


## 7. Run across three TSP sizes

We sweep TSP at **10, 30, 50** nodes - small / medium / large. Same
training settings everywhere so the only thing changing is the inner
problem size.

Per size we (1) generate data, (2) build both datasets, (3) train SPO+,
(4) clear the GPU cache, (5) train CaVE+, (6) record final regret and
wall-clock time.

Expect this to take ~10 minutes total - most of it is SPO+ training on
TSP-50, where every gradient step solves a 50-node TSP.


In [ ]:
SIZES = [10, 30, 50]
results = {}  # size -> {"spo": (regret, wall), "cave": (regret, wall)}

for n in SIZES:
    print(f"\n=== TSP-{n} ===", flush=True)

    # 1. generate data
    x_train, c_train = pyepo.data.tsp.genData(NUM_DATA, NUM_FEAT, n,
                                              deg=DEG, noise_width=NOISE, seed=42)
    x_test,  c_test  = pyepo.data.tsp.genData(NUM_TEST, NUM_FEAT, n,
                                              deg=DEG, noise_width=NOISE, seed=43)
    optmodel = tspDFJModel(num_nodes=n)

    # 2. build datasets (one-time cost, not included in training time)
    t = time.time()
    ds_train = optDataset(optmodel, x_train, c_train)
    ds_test  = optDataset(optmodel, x_test,  c_test)
    print(f"  optDataset       built in {time.time() - t:.1f}s")
    t = time.time()
    ds_train_c = optDatasetConstrs(optmodel, x_train, c_train)
    print(f"  optDatasetConstrs built in {time.time() - t:.1f}s")

    loader_spo  = DataLoader(ds_train,   batch_size=BATCH, shuffle=True)
    loader_test = DataLoader(ds_test,    batch_size=BATCH, shuffle=False)
    loader_cave = DataLoader(ds_train_c, batch_size=BATCH, shuffle=True,
                             collate_fn=collate_tight_constraints)

    # 3. SPO+
    spo_regrets, spo_wall = train_spo(optmodel, loader_spo, loader_test, NUM_FEAT)
    print(f"  SPO+   wall = {spo_wall:6.1f}s  final regret = {spo_regrets[-1]:.4f}")

    # 4. free SPO+ tensors before CaVE+
    if device.type == "cuda":
        torch.cuda.empty_cache()

    # 5. CaVE+
    cave_regrets, cave_wall = train_cave(optmodel, loader_cave, loader_test, NUM_FEAT)
    print(f"  CaVE+  wall = {cave_wall:6.1f}s  final regret = {cave_regrets[-1]:.4f}")
    print(f"  speedup (SPO+ wall / CaVE+ wall) = {spo_wall / cave_wall:.2f}x")

    results[n] = {"spo": (spo_regrets, spo_wall), "cave": (cave_regrets, cave_wall)}


## 8. Wall-clock comparison

CaVE+ stays roughly flat as the TSP grows; SPO+ does not, because every
step pays for a fresh TSP solve. We plot on a **log y-axis** because the
gap quickly spans an order of magnitude.


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))

x = np.arange(len(SIZES))
width = 0.35

spo_walls  = [results[n]["spo"][1]  for n in SIZES]
cave_walls = [results[n]["cave"][1] for n in SIZES]
speedups   = [s / c for s, c in zip(spo_walls, cave_walls)]

ax.bar(x - width / 2, spo_walls,  width, label="SPO+",  color="#4C72B0")
bars = ax.bar(x + width / 2, cave_walls, width, label="CaVE+", color="#DD8452")

for bar, sp in zip(bars, speedups):
    ax.annotate(f"{sp:.1f}x faster",
                xy=(bar.get_x() + bar.get_width() / 2, bar.get_height()),
                xytext=(0, 4), textcoords="offset points",
                ha="center", va="bottom", fontsize=10, fontweight="bold")

ax.set_yscale("log")
ax.set_xticks(x)
ax.set_xticklabels([f"TSP-{n}" for n in SIZES])
ax.set_ylabel("Wall-clock training time (s, log scale)")
ax.set_title(f"SPO+ vs CaVE+ training time ({EPOCHS} epochs, n_train={NUM_DATA})")
ax.legend()
ax.grid(axis="y", which="both", alpha=0.3)
plt.tight_layout(); plt.show()


## 9. Regret comparison

The point of CaVE is to be faster **without losing decision quality**. So
we also plot the final test regret. They should be within noise of each
other.


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))

spo_reg  = [results[n]["spo"][0][-1]  for n in SIZES]
cave_reg = [results[n]["cave"][0][-1] for n in SIZES]

ax.bar(x - width / 2, spo_reg,  width, label="SPO+",  color="#4C72B0")
ax.bar(x + width / 2, cave_reg, width, label="CaVE+", color="#DD8452")

ax.set_xticks(x)
ax.set_xticklabels([f"TSP-{n}" for n in SIZES])
ax.set_ylabel("Final test regret")
ax.set_title(f"Final regret after {EPOCHS} epochs (lower is better)")
ax.legend()
ax.grid(axis="y", alpha=0.3)
plt.tight_layout(); plt.show()


## 10. Takeaways

- **CaVE skips the per-step solve.** For binary LPs, the cone of binding
  normals at the true optimum is enough supervision.
- **The speedup grows with problem size.** SPO+ pays an inner TSP solve
  every batch; CaVE+ pays the same constant-time cone projection.
- **Quality holds.** Final regret tracks SPO+ within noise at all three
  sizes.
- **GPU matters.** The APGD projection is `torch.compile`'d for CUDA -
  run CaVE on GPU.
- **Drop-in usage:** `pyepo.func.coneAlignedCosine(optmodel)` with
  `pyepo.data.dataset.optDatasetConstrs` + `collate_tight_constraints`.
  That's the whole API surface.
